In [5]:
import re
import math
from collections import defaultdict

def parse_file_content(text_content):
    # Use double quotes for the regex pattern to handle the single quote correctly
    clean_text = re.sub(r"'", "", text_content)

    raw_tokens = clean_text.split()
    sentences = []
    current_sentence = []

    for token in raw_tokens:
        if '_' not in token:
            continue
        word, tag = token.rsplit('_', 1)
        current_sentence.append((word, tag))
        if word in ['.', '?', '!'] and tag == '.':
            sentences.append(current_sentence)
            current_sentence = []

    if current_sentence:
        sentences.append(current_sentence)

    return sentences

class HMM_POS_Tagger:
    def __init__(self):
        self.transitions = defaultdict(lambda: defaultdict(int))
        self.emissions = defaultdict(lambda: defaultdict(int))
        self.tag_counts = defaultdict(int)
        self.vocab = set()
        self.tags = set()
        self.start_tag = '<s>'
        self.end_tag = '</s>'
        self.trans_probs = {}
        self.emit_probs = {}

    def train(self, sentences):
        self.vocab = set()
        self.tags = set()
        self.tag_counts = defaultdict(int)
        self.transitions = defaultdict(lambda: defaultdict(int))
        self.emissions = defaultdict(lambda: defaultdict(int))

        for sentence in sentences:
            prev_tag = self.start_tag
            self.tags.add(self.start_tag)

            for word, tag in sentence:
                self.vocab.add(word)
                self.tags.add(tag)
                self.transitions[prev_tag][tag] += 1
                self.emissions[tag][word] += 1
                self.tag_counts[tag] += 1
                prev_tag = tag

            self.transitions[prev_tag][self.end_tag] += 1
            self.tag_counts[self.start_tag] += 1

        self._calculate_probabilities()

    def _calculate_probabilities(self):
        alpha = 1e-5
        all_tags = list(self.tags)
        all_tags.append(self.end_tag)

        for t1 in self.tags:
            total_trans = sum(self.transitions[t1].values())
            for t2 in all_tags:
                count = self.transitions[t1][t2]
                prob = (count + alpha) / (total_trans + alpha * len(all_tags))
                self.trans_probs[(t1, t2)] = math.log(prob)

        for tag in self.tags:
            total_emit = sum(self.emissions[tag].values())
            for word in self.emissions[tag]:
                count = self.emissions[tag][word]
                prob = (count + alpha) / (total_emit + alpha * (len(self.vocab) + 1))
                self.emit_probs[(tag, word)] = math.log(prob)

            self.emit_probs[(tag, '<UNK>')] = math.log(alpha / (total_emit + alpha * (len(self.vocab) + 1)))

    def viterbi(self, sentence_words):
        n_length = len(sentence_words)
        if n_length == 0:
            return []

        viterbi_matrix = []
        backpointer = []
        valid_tags = [t for t in self.tags if t != self.start_tag]

        first_col = {}
        first_bp = {}
        word0 = sentence_words[0]

        for tag in valid_tags:
            trans_p = self.trans_probs.get((self.start_tag, tag), -1000)
            if word0 in self.vocab:
                emit_p = self.emit_probs.get((tag, word0), -1000)
            else:
                emit_p = self.emit_probs.get((tag, '<UNK>'), -1000)
            first_col[tag] = trans_p + emit_p
            first_bp[tag] = self.start_tag

        viterbi_matrix.append(first_col)
        backpointer.append(first_bp)

        for t in range(1, n_length):
            curr_col = {}
            curr_bp = {}
            word = sentence_words[t]

            for curr_tag in valid_tags:
                if word in self.vocab:
                    emit_p = self.emit_probs.get((curr_tag, word), -1000)
                else:
                    emit_p = self.emit_probs.get((curr_tag, '<UNK>'), -1000)

                best_prev_prob = float('-inf')
                best_prev_tag = None

                for prev_tag in valid_tags:
                    trans_p = self.trans_probs.get((prev_tag, curr_tag), -1000)
                    prev_path_prob = viterbi_matrix[t-1][prev_tag]
                    prob = prev_path_prob + trans_p + emit_p

                    if prob > best_prev_prob:
                        best_prev_prob = prob
                        best_prev_tag = prev_tag

                curr_col[curr_tag] = best_prev_prob
                curr_bp[curr_tag] = best_prev_tag

            viterbi_matrix.append(curr_col)
            backpointer.append(curr_bp)

        best_final_prob = float('-inf')
        best_last_tag = None

        for tag in valid_tags:
            end_trans = self.trans_probs.get((tag, self.end_tag), -1000)
            prob = viterbi_matrix[n_length-1][tag] + end_trans
            if prob > best_final_prob:
                best_final_prob = prob
                best_last_tag = tag

        best_path = [best_last_tag]
        for t in range(n_length - 1, 0, -1):
            best_tag = best_path[-1]
            prev_tag = backpointer[t][best_tag]
            best_path.append(prev_tag)

        return list(reversed(best_path))

def get_k_folds(data, k=3):
    n = len(data)
    fold_size = math.ceil(n / k)
    folds = []
    for i in range(0, n, fold_size):
        folds.append(data[i:i + fold_size])
    return folds

def calculate_metrics(y_true_flat, y_pred_flat, all_tags):
    true_positives = defaultdict(int)
    false_positives = defaultdict(int)
    false_negatives = defaultdict(int)
    support = defaultdict(int)

    for t, p in zip(y_true_flat, y_pred_flat):
        support[t] += 1
        if t == p:
            true_positives[t] += 1
        else:
            false_negatives[t] += 1
            false_positives[p] += 1

    total_support = len(y_true_flat)
    weighted_precision = 0
    weighted_recall = 0
    weighted_f1 = 0

    for tag in support.keys():
        tp = true_positives[tag]
        fp = false_positives[tag]
        fn = false_negatives[tag]

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        if (precision + recall) > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0

        weight = support[tag] / total_support
        weighted_precision += precision * weight
        weighted_recall += recall * weight
        weighted_f1 += f1 * weight

    return weighted_precision, weighted_recall, weighted_f1

def main():
    file_path = '/content/wsj_pos_tagged_en.txt'
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return

    all_sentences = parse_file_content(content)
    print(f"Total Sentences Parsed: {len(all_sentences)}")

    K = 3
    folds = get_k_folds(all_sentences, K)

    overall_precision = 0
    overall_recall = 0
    overall_f1 = 0

    print(f"\nStarting {K}-Fold Cross-Validation...")

    for i in range(K):
        print(f"\nProcessing Fold {i+1}/{K}...")

        test_data = folds[i]
        train_data = []
        for j in range(K):
            if i != j:
                train_data.extend(folds[j])

        hmm = HMM_POS_Tagger()
        hmm.train(train_data)

        y_true = []
        y_pred = []

        for sentence in test_data:
            words = [pair[0] for pair in sentence]
            true_tags = [pair[1] for pair in sentence]

            predicted_tags = hmm.viterbi(words)

            if len(predicted_tags) == len(true_tags):
                y_true.extend(true_tags)
                y_pred.extend(predicted_tags)

        p, r, f1 = calculate_metrics(y_true, y_pred, hmm.tags)

        print(f"Fold {i+1} Results -> Precision: {p:.4f}, Recall: {r:.4f}, F1-Score: {f1:.4f}")

        overall_precision += p
        overall_recall += r
        overall_f1 += f1

    print("\n" + "="*30)
    print(f"Average Precision: {overall_precision / K:.4f}")
    print(f"Average Recall:    {overall_recall / K:.4f}")
    print(f"Average F1-Score:  {overall_f1 / K:.4f}")
    print("="*30)

if __name__ == "__main__":
    main()

Total Sentences Parsed: 3874

Starting 3-Fold Cross-Validation...

Processing Fold 1/3...
Fold 1 Results -> Precision: 0.9043, Recall: 0.8616, F1-Score: 0.8772

Processing Fold 2/3...
Fold 2 Results -> Precision: 0.9009, Recall: 0.8628, F1-Score: 0.8752

Processing Fold 3/3...
Fold 3 Results -> Precision: 0.9124, Recall: 0.8731, F1-Score: 0.8867

Average Precision: 0.9059
Average Recall:    0.8659
Average F1-Score:  0.8797
